# 3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

o find_nascentes quase que deu certo, mas tem um erros e antes de eu resolver esses erros eu preciso encontrar onde eles estõ acontecendo... bom, vamos começar usando um específico como exemplo, o do CORREGO MANDAQUI

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import os
#from tqdm import tqdm


In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

# Raw gdf

In [3]:
drenageo = gpd.read_file(
    os.path.join(
        'data',
        'drenagem.zip'
    )
)

# Silver gdf

In [4]:
## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)
## Conferir se todos os 'cd_tipo_cu' sejam do mesmo tipo
drenageo['cd_tipo_cu'].dtype

dtype('float64')

# Determinar Correntes Estimadas (só por enquanto, dps vamos usar o do Elias e tals) 

In [5]:
drenageo.sample(10)
#* 11: trecho em estado natural
#* 12: lago ou reservatório
#* 10: trecho fechado
#* 9: trecho a céu aberto

cus_to_keep = [9.0, 11.0]
colors_dictionarie= {
    9.0 : 'turquoise',
    11.0 : 'aquamarine',
    10.0 : 'pink',
    12.0 : 'pink',
}

drenageo['colors'] = drenageo['cd_tipo_cu'].map(colors_dictionarie)

# Create GDF copy

In [6]:
drena= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry',
    'nm_bairro'
]]

# Erro 1: Pontos buff

In [7]:
erro_pontos = drena.copy()
erro_pontos_buff = drena.copy()
erro_pontos['geometry']=shapely.get_point(drena.geometry, -1)

erro_pontos_buff['geometry'] = erro_pontos['geometry'].buffer(10)

Que engraçado. Encontramos o erro e ele tá aqui em cima, não lá embaixo, onde eu esperava.

Minha teoria: o shapely só pega uma das extremidades por ponto, não duas, como eu acho que deveria ser o certo. 

Descobrimos!!! Aaaah, o ponto é que se passar o `get_point()` com `0`, aí, vai retornar o primeiro ponto da linha e com o `-1` vai retornar o último ponto da linha... por isso que não estava dando certo.

# V.3. Agora nosso problema está sendo (representado por) essa VILA BARBOSA:

Após conferir isso lá embaixo, realmente o VILA BARBOSA não tem nada nela, então precisamos descobrir o motivo.

In [8]:
teste=drenageo.loc[drenageo['cd_identif']==1649]
ponteste = shapely.get_point(teste.geometry, 0)
ponteste_fim = shapely.get_point(teste.geometry, -1)

In [9]:
gdf_ponto = gpd.GeoDataFrame(geometry=ponteste)
gdf_ponto.set_crs(teste.crs, inplace=True)  # usa o CRS do original, se definido

gdf_ponfim = gpd.GeoDataFrame(geometry=ponteste_fim)
gdf_ponfim.set_crs(teste.crs, inplace=True)

,geometry
1771,POINT (329606.913 7400740.117)


In [ ]:
## visualizar teste

m= teste.explore(color='orange')
gdf_ponto.explore(
    m=m
)

gdf_ponfim.explore(
    m=m,
    color='purple'
)

In [10]:
pontos = drena.copy()
pontos_0 = drena.copy()
pontos_1 = drena.copy()

pontos_0['geometry']=shapely.get_point(drena.geometry, 0)
pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
pontos_1['geometry'] = shapely.get_point(drena.geometry, -1)
pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"
pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)

pontos_buff = pontos.copy()
pontos_buff['geometry'] = pontos['geometry'].buffer(10)

## visualizar 2.0

#### já deu erro no pontos_buff.explore()
#### mas o pontos.explore() tá certo!! (foi burrice minha)
#### Vamos tentar agr com 1
m= pontos.explore()
teste.explore(m=m, color='red')
### oxi, o ponto 1 já ta mostrando os dois... deixa eu ver se isso acontece com o ponto 0
##### ENTÃO: o que eu to vendo ali que eu to achando que são 2 pontos, na vdd é o ponto final da linha + ponto final da linha vizinha
### NO 0 só aparece um... 


## (DONE) Tarefa:
comparar os shapes/ len() em: 
* pontos
* ponto_0
* ponto_1
Precisamos levar em consideração o shape das linhas... deveria ter o dobro, certo?

In [11]:
print(
    f'Shape das linhas: {drena.shape}\n'
    + f'O dobro disso é: {drena.shape[0]*2}'
)
print(
    f'Shape dos pontos_0: {pontos_0.shape}\n'
     + 'Conclusão: pontos_0 só retorna um ponto por linha'
)
print(
    f'Shape dos pontos_1: {pontos_1.shape}\n'
    + 'Conclusão: pontos_1 também só retorna um ponto por linha TAMBÉM'
)
print(
    f'Shape dos pontos: {pontos.shape}\n'
    + f'Shape dos pontos_buff: {pontos_buff.shape}\n'
    + f'Eles são iguais? {pontos.shape[0]==pontos_buff.shape[0]}\n'
    + 'Conclusão: PARECE que em pontos aparece tudo... vamos voltar na visualização pra ver se isso tá certo'
    + '\n(spoiler: está, eu que fui burra)'
    
)



Shape das linhas: (27611, 6)
O dobro disso é: 55222
Shape dos pontos_0: (27611, 7)
Conclusão: pontos_0 só retorna um ponto por linha
Shape dos pontos_1: (27611, 7)
Conclusão: pontos_1 também só retorna um ponto por linha TAMBÉM
Shape dos pontos: (55222, 7)
Shape dos pontos_buff: (55222, 7)
Eles são iguais? True
Conclusão: PARECE que em pontos aparece tudo... vamos voltar na visualização pra ver se isso tá certo
(spoiler: está, eu que fui burra)


Ok, acho que agora tá tudo corrigido
## Qual foi a burrada?
Ao invés de fazer o pontos_buff a partir dos pontos, eu estava fazendo a partir do drenageo, então eles só pegavam alguns dos pontos, mudando a geometria das linhas pros primeiros pontos respectivos. Ai, burrada.
## Correção:
`pontos_buff = pontos.copy()` 

# v.4. Agora o erro está em trechos que não foram pegos
Minha teoria:
1. O buffer é muito grande. Então só precisamos ir testando buffers menores e ir vendo se o quando o shape aumenta e se aumenta mt e se não perde nada
Registros de shape:
* 10m: 44903
* 5m: 44586 

## Recorte geográfico
O Mauryas tinha sugerido fazer um recorte por distritos... eu vou fazer por bairros, pra não ter que fazer um overlay.

In [12]:
drenageo.sample()

,cd_identif,cd_tipo_ac,tx_tipo_ac,cd_numero_,nm_bairro,nm_acident,qt_comprim,cd_tipo_cu,nm_tipo_cu,nm_via_pro,nm_descrit,nm_tipo_tr,dt_atualiz,cd_usuario,geometry,colors
1134,1048.0,COR,CORREGO,2,JARDIM IPANEMA,CORREGO DA FAZENDA,11.730574,10.0,Trecho canalizado subterrâneo,CRISTOVAO LOPES,Córrego da Fazenda,Trecho fechado,2025-01-03,None,"LINESTRING (346373.632 7393857.325, 346385.292...",pink


In [13]:
teste_v4 = drenageo.loc[drenageo['cd_identif']==10226.0]

In [14]:
bairros =drenageo.loc[drenageo['nm_acident'].str.contains('GOLFE'), 'nm_bairro'].unique()
recort = drena.loc[drena['nm_bairro'].isin(bairros)]

Como eu vou precisar fazer a comparação de dois gdfs em situações semelhantes, vou transformar os processos em uma função

# Função drop_intersec

In [15]:
# Função drop_intersec

def drop_intersec(pontos, linhas):
    intersecs_bool=[]

    for i, row in pontos.iterrows():
        outras_geoms = pontos.loc[pontos.index!=i]
        intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
        if len(intersecs_bool.loc[intersecs_bool==True])<1:
            pontos.loc[pontos.index==i, 'intersec_bool'] = False
        else:
            pontos.loc[pontos.index==i, 'intersec_bool'] = True
    
    for i, row in pontos.loc[pontos['intersec_bool']==False].iterrows():
        outras_linhas = linhas.loc[linhas['cd_identif']!=row['cd_identif']]
        intersecs_bool = row.geometry.intersects(outras_linhas.geometry)
        if len(intersecs_bool.loc[intersecs_bool==True])<1:
            pontos.loc[pontos.index==i, 'intersec_bool']=False
        else:
            pontos.loc[pontos.index==i, 'intersec_bool'] = True

    pontos = pontos.loc[pontos['intersec_bool']==False]
    

    return pontos

### Funções dos pontos
Como vou repetir muito o processo, eu vou só chamar uma função, que é mais fácil

In [16]:
#funções dos pontos

def get_pontos(gdf):
    pontos = gdf.copy()
    pontos_0 = gdf.copy()
    pontos_1 = gdf.copy()

    pontos_0['geometry']=shapely.get_point(gdf.geometry, 0)
    pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
    pontos_1['geometry'] = shapely.get_point(gdf.geometry, -1)
    pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"
    pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)
    
    return pontos
    
def get_pontos_buff(
    #gdf,
    pontos,
    buffer
):
    #pontos = get_pontos(gdf)
    pontos_buff = pontos.copy()
    pontos_buff['geometry'] = pontos['geometry'].buffer(buffer)
    return pontos_buff


def intersec_buffed(gdf, pontos, buffer):
    buffed=get_pontos_buff(pontos, buffer)
    final = drop_intersec(pontos=buffed, linhas=gdf)
    return final
    

In [17]:
recort_pontos = get_pontos(recort)
buffer10 = intersec_buffed(recort, recort_pontos, 10)
buffer9 = intersec_buffed(recort, recort_pontos, 9)
buffer8=intersec_buffed(recort, recort_pontos,8)
buffer7=intersec_buffed(recort, recort_pontos,7)
buffer6=intersec_buffed(recort, recort_pontos,6)
buffer5 = intersec_buffed(recort, recort_pontos, 5)

In [18]:
print(
    f'Shape das linhas: {recort.shape}\n'
    + f'O dobro disso é: {recort.shape[0]*2}'
)
print(
    f'Shape dos pontos: {recort_pontos.shape}'
)
print(f'''
PONTOS:
    10: {buffer10.shape}
    9: {buffer9.shape}
    8: {buffer8.shape}
    7: {buffer7.shape}
    6: {buffer6.shape}
    5: {buffer5.shape}


''')

print("O problema é que agora eu não lembro se era melhor sobrar ou faltar coisa kkkkkk bom vou almoçar e na volta a gente vê como ficou")

Shape das linhas: (9120, 6)
O dobro disso é: 18240
Shape dos pontos: (18240, 7)

PONTOS:
    10: (4375, 8)
    9: (4413, 8)
    8: (4448, 8)
    7: (4489, 8)
    6: (4527, 8)
    5: (4571, 8)



O problema é que agora eu não lembro se era melhor sobrar ou faltar coisa kkkkkk bom vou almoçar e na volta a gente vê como ficou


O que eu acho que seria bom fazer agr. Abrir o arquivo salvo dos 10 m de buffer em outro notebook
 e ir explorando os pontos (sem buffer) dando loc[] por id dos buffer10-buffer5
 ou só ir dando loc com isin e ver quais são os que alteram menos, mas adicionam o que tinha ficado de fora
 ou ver qual não deixa de pegar nada... AI NÃO SEI

In [19]:
pontos_buff = drop_intersec(pontos_buff, drena)

In [20]:
pontos_buff.loc[pontos_buff['intersec_bool']!=False]

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,nm_bairro,cd_point,intersec_bool


In [21]:
pontos_buff.shape

(10319, 8)

In [22]:
pontos_buff['cd_tipo_cu'].astype(dtype='float', copy=False)
pontos_buff= pontos_buff.loc[pontos_buff['cd_tipo_cu'].isin(cus_to_keep)]

In [23]:
pontos_buff.shape

(8382, 8)

In [ ]:
# Visualizar
### Ok, eu estava errada... mesmo com o intersec e um mega buffer nos pontos, ainda não dá certo, vamos voltar pra tatica do Henrique mesmo
m= drenageo.explore(color='pink')

#drenageo.loc[drenageo['cd_tipo_cu'].isin(cus_to_keep)].explore(m=m,color='purple')

#drenageo.loc[drenageo['nm_acident']=="CORREGO DOS FORNOS"].explore(m=m, color='orange')

teste_v4.explore(m=m, color='red')

pontos_buff.explore(
    m=m,
    color="green"
)

E depois disso tudo, aquele bendito riozinho que o Mauryas achou ainda não foi pego! Vou fazer um commit pro que eu já fiz e depois passo as coreeções daqui lá pro find_nascentes, e depois procurar os motivos desse novo erro

In [24]:
# Salvar arquivo para a validação do Mauryas
pontos_buff.to_file(
    os.path.join(
        'data',
        'pontos_buff_v4.0.geojson'
    ),
    driver="GeoJSON"
)